# TF-IDF From Scratch (No Libraries)
Preprocessing and TF-IDF feature extraction implemented in pure Python.

## 1. Imports

In [ ]:
import math
import re
import string

## 2. Stopwords, Protected Words & Normalize Map

In [ ]:
STOPWORDS = {
    "the", "is", "in", "it", "of", "and", "a", "an", "to", "was",
    "for", "on", "are", "with", "as", "at", "be", "by", "this",
    "that", "have", "had", "not", "but", "from", "or", "he", "she",
    "they", "we", "you", "i", "do", "did", "will", "would", "can",
    "could", "should", "may", "might", "its", "it's", "has",
    "no", "so", "up", "all", "also", "about", "before",
    "after", "where", "while", "their", "these", "every",
    "who", "what", "were", "our", "them", "now", "how",
    "just", "same", "since", "back", "down", "through",
    "without", "one", "here", "last", "next", "new",
    "being", "said", "told", "went", "came",
}

PROTECTED_WORDS = {
    "missile", "region", "regime", "president", "reform", "regional",
    "release", "remain", "remove", "disrupt", "dismantle",
    "display", "distance", "district", "mission", "press",
    "pressure", "progress", "precise", "prevent", "process",
    "declaration", "cooperation", "department", "decrease",
    "carrier", "cooper", "provide", "information", "organization",
    "generation", "production", "transition", "operation", "operations"
}

# normalize post-stem forms that are still broken
NORMALIZE = {
    "provid"     : "provide",     # post-stem form of "provided"
    "decreas"    : "decrease",    # post-stem form of "decreased"
    "departments": "department",  # plural not caught by stemmer
}

prefixes= ["un", "re", "pre", "dis", "mis", "over", "out", "sub", "inter", "non"]
suffixes= ["ing", "ed", "ly", "er", "tion", "sion", "ness", "ment", "ful", "less", "able", "ible", "s"]

## 3. Stemmer Function

In [ ]:
def stem(word):
    if word in PROTECTED_WORDS:
        return word
    for suffix in suffixes:
        if word.endswith(suffix) and len(word) - len(suffix) >= 4:
            return word[:-len(suffix)]
    for prefix in prefixes:
        if word.startswith(prefix) and len(word) - len(prefix) >= 4:
            return word[len(prefix):]
    return word

## 4. Preprocessing Function

In [ ]:
def preprocess(corpus):
    all_tokens= []

    for i, doc in enumerate(corpus):

        doc= doc.lower()                                                # lowercase

        doc= re.sub(r'\d+', '', doc)                                   # remove numbers

        doc= re.sub(r"'s\b", '', doc)                                  # remove possessive 's

        doc= doc.replace('-', ' ').replace('\u2014', ' ').replace('\u2026', ' ') # remove hyphens, em dash, ellipsis

        doc= doc.translate(str.maketrans('', '', string.punctuation))  # remove punctuation

        doc= re.sub(r'\s+', ' ', doc).strip()                          # remove extra whitespace

        tokens= doc.split()                                             # tokenize

        tokens= [word for word in tokens if word not in STOPWORDS]     # remove stopwords

        # stemming
        stemmed_tokens= []
        for word in tokens:
            stemmed_tokens.append(stem(word))
        tokens= stemmed_tokens

        tokens= [NORMALIZE.get(word, word) for word in tokens]         # normalize irregular forms

        tokens= [word for word in tokens if len(word) >= 3]            # remove short words

        all_tokens.append(tokens)

    return all_tokens

## 5. TF, IDF, and TF-IDF Functions

In [ ]:
def compute_tf(tokens):
    tf= {}
    total= len(tokens)
    for word in tokens:
        tf[word]= tf.get(word, 0) + 1
    for word in tf:
        tf[word]= tf[word] / total
    return tf


def compute_idf(all_tokens):
    N= len(all_tokens)
    idf= {}
    for tokens in all_tokens:
        for word in set(tokens):
            idf[word]= idf.get(word, 0) + 1
    for word in idf:
        idf[word]= math.log((1 + N) / (1 + idf[word])) + 1
    return idf


def compute_tfidf(tf, idf):
    tfidf= {}
    for word in tf:
        tfidf[word]= tf[word] * idf.get(word, 0)
    return tfidf

## 6. Corpus

In [ ]:
corpus= [
    """This Week in DOW: Delivering 'Shock and Awe' to Iran, Defense Leaders' Declaration, Updates on AI On Feb. 28, the U.S. military commenced Operation Epic Fury under the direct order of President Donald J. Trump. "We are delivering twice the airpower of shock and awe and seven times the intensity of Israel's 12-day war," Pentagon Press Secretary Kingsley Wilson said today in the War Department's Weekly Sitrep video. "I stand before you today with one unmistakable message about Operation Epic Fury: America is winning decisively, devastatingly and without mercy," Secretary of War Pete Hegseth told the media yesterday. "We are destroying their missiles and demolishing their missile industry. We are annihilating their navy. Every warship, fast attack craft and naval base is being hunted down," Wilson said. Iran's entire nuclear program is being systematically terminated. However, Wilson said, the goal is not nation-building; it is pure American dominance — enforcing peace through strength. "We are ensuring the regime's terrorist proxies can no longer destabilize the region or attack our forces with [improvised explosive devices] and roadside bombs that have killed and wounded thousands of Americans, and we are guaranteeing Iran will never obtain a nuclear weapon," she said, adding, "Epic Fury is here to finish the job." """,
    """Wilson said the department honors the six American heroes who made the ultimate sacrifice while supporting the operation. "Their fearless service will never be forgotten. We will avenge them by destroying every missile, every ship and every last trace of the regime that dared to strike our forces. These patriots paid the price in full, and the Department of War will deliver total victory in their name. Their sacrifice fuels our fight, and we will not rest until the enemy is defeated," she said. Yesterday, Hegseth traveled to Doral, Florida, to host the inaugural Americas Counter Cartel Conference at U.S. Southern Command headquarters. Like-minded regional defense and security leaders from 17 nations in the Caribbean, Central America and South America attended the conference, where they signed a historic joint security declaration, she said. The declaration affirmed that the department is strengthening its cooperation with partners across the Western Hemisphere to counter narco-terrorist networks — a move intended to bolster regional security, economic stability and the safety of partner nations. The declaration aims to expand cooperation and information sharing, disrupt cartel operations, secure borders and infrastructure, and protect people.""",
    """Also yesterday, Hegseth visited U.S. Central Command headquarters in Tampa, Florida. During the visit, he met with Centcom Commander Navy Adm. Brad Cooper for updates on Operation Epic Fury. Cooper said Iran's ballistic missile attacks have decreased by 90% since the first day of the conflict, and Iranian drone attacks have decreased by 83% in the same time frame. At sea, he said the count of sunken Iranian navy ships has surpassed 30, noting that just before the press conference, Centcom forces hit an Iranian drone carrier ship, roughly the size of a World War II aircraft carrier. Under orders from Trump, Centcom forces are working to destroy Iran's missile industrial base. "We're not just hitting what they have, we're destroying their ability to rebuild. And so, as we transition to the next phase of this operation, we will systemically dismantle Iran's missile production capability for the future, and that's absolutely in progress," Cooper said.""",
    """Wilson provided an update on the department's usage of artificial intelligence following the president's order for every federal agency to cease use of technology provided by Anthropic immediately. "America's warfighters … will never be held hostage by unelected tech executives and Silicon Valley ideology. We will decide, we will dominate and we will win," Wilson said. Last week, Hegseth delivered an update on the War Department's partnership with Scouting America, formerly known as the Boy Scouts of America. "After putting them on notice, they stepped up and promised reform," Wilson said, noting that the organization has now committed to eliminating its diversity, equity and inclusion efforts, scrapping the citizenship and society merit badge, basing all membership strictly on biological sex at birth, introducing a new military service merit badge and waiving fees for children of active-duty troops. The department will hold a six-month compliance review to ensure these promises become permanent. "This is how we raise the next generation of strong, merit-driven, patriotic Americans who will one day defend this nation — back to basics, back to excellence," she said."""
]

## 7. Run Preprocessing

In [ ]:
all_tokens= preprocess(corpus)

for i, tokens in enumerate(all_tokens):
    print(f"Document {i+1} tokens:")
    print(tokens)
    print()

## 8. Compute TF-IDF & Display Results

In [ ]:
idf= compute_idf(all_tokens)

for i, tokens in enumerate(all_tokens):
    tf= compute_tf(tokens)
    tfidf= compute_tfidf(tf, idf)

    # sort by score descending
    sorted_tfidf= sorted(tfidf.items(), key=lambda x: x[1], reverse=True)

    print(f"\nDocument {i+1}:")
    print(f"  {'Word':<20} {'TF-IDF Score':<15}")
    print(f"  {'-'*35}")
    for word, score in sorted_tfidf[:10]:
        print(f"  {word:<20} {score:.6f}")